# Figure 4: spatial distribution of key features

- **Paper:** Fig 4
- **Original notebook:** `40_map_of_key_features (1) (1).ipynb`
- **Reads:**
  - `outputs/school_related_pre.pkl`
  - `outputs/school_related_post.pkl (OLD 2020-22 file, see TODO.md)`
  - `data/raw/geographic_boundaries/counties`
- **Writes:**
  - `outputs/plots/keyfeatures.tiff`

> Note: Taken from the latest version, `40_map_of_key_features (1) (1).ipynb` (cividis colormap, saves keyfeatures.tiff). Caption says pre-COVID, but this code averages pre and post. See TODO.md.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup


import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import xgboost as xgb
#from sklearn.metrics import aic
import graphviz




from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

# pip install rioxarray  # see requirements.txt
# pip install pysal  # see requirements.txt
# pip install contextily  # see requirements.txt
import libpysal


import rioxarray  # surface data manipulation
import xarray  # surface data manipulation
from pysal.explore import esda  # spatial stats
from pysal.lib import weights  # weights
import contextily



In [ ]:
# pip install optuna  # see requirements.txt
import optuna

# pip install shap  # see requirements.txt
import shap

In [ ]:
# load data
# import data
random.seed(123)


# cumulative enrollment
# total_enrolled = pd.read_csv(f"{DATA_DIR}/school_related/CEnroll2021.txt",sep="\t",encoding='latin-1')

school_pre = pd.read_pickle(f"{OUTPUT_DIR}/school_related_pre.pkl")
school_pre['origin'] = 'pre'

rows_pre, cols_pre = school_pre.shape
print(f"Number of rows_pre: {rows_pre}")
print(f"Number of cols_pre: {cols_pre}")

school_post = pd.read_pickle(f'{OUTPUT_DIR}/school_related_post.pkl')
school_post['origin'] = 'post'

rows_post, cols_post = school_post.shape
print(f"Number of rows_post: {rows_post}")
print(f"Number of cols_post: {cols_post}")


school_related= pd.concat([school_pre,school_post])
rows, cols = school_related.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {cols}")

In [ ]:
# Compare column names between school_pre and school_post
pre_cols = set(school_pre.columns)
post_cols = set(school_post.columns)
combined_col = set(school_related)

only_in_pre = pre_cols - post_cols
only_in_post = post_cols - pre_cols
in_combined = combined_col - pre_cols

print("Columns only in school_pre:", only_in_pre)
print("Columns only in school_post:", only_in_post)
print(in_combined)

In [ ]:
print(combined_col)
school_related

In [ ]:
school_related = school_related.groupby(['SchoolName', 'SchoolCode']).mean(numeric_only=True).reset_index()

relevant  = school_related.drop(columns = ['SchoolCode','SchoolName',"DistrictCode"])#'DistrictName','closest_UC','closest_CSU'])
# relevant = school_related[['college_going_rate', 'YearsTeaching', 'years_teaching_std',
#        'ChronicAbsenteeismRate', 'pct_frpm', 'pct_passed_science',
#        'college_going_rate', 'pct_passed_english', 'school_expenditures',
#        'ShannonIndex', 'ProportionMale', 'ProportionWhite', 'DistrictCode']]

# relevant = school_related
for col in relevant.columns:
    print(col)
    school_related[col] = pd.to_numeric(school_related[col], errors='coerce')

In [ ]:
school_related

In [ ]:
'''
# cleaning
# cleaned_data = school_related.dropna(subset=relevant.columns.values)

school_related = school_related.groupby(['SchoolName', 'SchoolCode']).mean(numeric_only=True).reset_index()
cleaned_data = school_related
'''

In [ ]:
print(school_pre.dtypes)


In [ ]:
gdf = gpd.GeoDataFrame(school_pre, geometry=gpd.points_from_xy(school_pre['Longitude'], school_pre['Latitude']), crs='EPSG:4326')

cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')


# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
# zi = (xi – min(x)) / (max(x) – min(x))
gdf[['college_going_rate',
             'ChronicAbsenteeismRate',
             'pct_frpm', 'AP_Count',
             "pct_passed_math"]] = gdf[['college_going_rate',
                                                     'ChronicAbsenteeismRate',
                                                     'pct_frpm', 'AP_Count',
                                                 "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.viridis  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  'FRPM'   , 'AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

    gdf.plot(column=col, ax=ax, markersize=5,
                     cmap=cmap, legend=False, alpha = 0.75)

    ax.set_axis_off()
    ax.set_title(title,loc='left')

cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array for ScalarMappable
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

plt.tight_layout
plt.show()


In [ ]:
gdf_post = gpd.GeoDataFrame(school_post, geometry=gpd.points_from_xy(school_post['Longitude'], school_post['Latitude']), crs='EPSG:4326')

cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf_post[cols] = gdf_post[cols].apply(pd.to_numeric, errors='coerce')


# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
# zi = (xi – min(x)) / (max(x) – min(x))
gdf_post[['college_going_rate',
             'ChronicAbsenteeismRate',
             'pct_frpm', 'AP_Count',
             "pct_passed_math"]] = gdf_post[['college_going_rate',
                                                     'ChronicAbsenteeismRate',
                                                     'pct_frpm', 'AP_Count',
                                                 "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.viridis  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  'FRPM'   , 'AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

    gdf_post.plot(column=col, ax=ax, markersize=5,
                     cmap=cmap, legend=False, alpha = 0.75)

    ax.set_axis_off()
    ax.set_title(title,loc='left')

cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array for ScalarMappable
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

plt.tight_layout
plt.show()


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# 1. Create GeoDataFrame
gdf = gpd.GeoDataFrame(
    school_pre,
    geometry=gpd.points_from_xy(school_pre['Longitude'], school_pre['Latitude']),
    crs='EPSG:4326'
)

# 2. Clean numeric columns
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# 3. Normalize values
gdf[cols] = gdf[cols].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

# 4. Load base map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

# 5. Plot
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

cmap = mpl.cm.viridis
gridsize = 50  # adjust for resolution of hexes

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    # Plot counties background
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)

    # Plot hexbin
    ax.hexbin(
        gdf.geometry.x, gdf.geometry.y,
        C=gdf[col],
        gridsize=gridsize,
        reduce_C_function=np.mean,
        cmap=cmap,
        mincnt=1,   # only show hexes with at least 1 point
        linewidths=0.0,
        alpha=0.8
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# 6. Colorbar
cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])
norm = mpl.colors.Normalize(vmin=0, vmax=1)
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

plt.suptitle('Spatial Distribution of Key Features (Hexbin)', x=0.45, y=0.93, fontweight='bold')
plt.subplots_adjust(wspace=-0.3, hspace=-0.05)
plt.tight_layout()
plt.show()


In [ ]:
'''
Pre covid

with Jenks Natural Breaks classification

'''
gdf = gpd.GeoDataFrame(school_pre, geometry=gpd.points_from_xy(school_pre['Longitude'], school_pre['Latitude']), crs='EPSG:4326')

cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')


# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
# zi = (xi – min(x)) / (max(x) – min(x))
gdf[['college_going_rate',
             'ChronicAbsenteeismRate',
             'pct_frpm', 'AP_Count',
             "pct_passed_math"]] = gdf[['college_going_rate',
                                                     'ChronicAbsenteeismRate',
                                                     'pct_frpm', 'AP_Count',
                                                 "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

# ✅ ONLY THIS PART CHANGED: Jenks classification replaces normalization
import mapclassify
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col] = classifier.find_bin(gdf[col])


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.viridis  # Change this to your preferred colormap

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  'FRPM'   , 'AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

    gdf.plot(column=col, ax=ax, markersize=10,
            cmap=cmap, legend=False, alpha = 0.75,
            edgecolor='white', linewidth=0.2)


    ax.set_axis_off()
    ax.set_title(title,loc='left')

cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Empty array for ScalarMappable
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

plt.tight_layout
plt.show()


In [ ]:
'''
all years

with Jenks Natural Breaks classification



Ali
'''

# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')


# shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
# zi = (xi – min(x)) / (max(x) – min(x))
gdf[['college_going_rate',
             'ChronicAbsenteeismRate',
             'pct_frpm', 'AP_Count',
             "pct_passed_math"]] = gdf[['college_going_rate',
                                                     'ChronicAbsenteeismRate',
                                                     'pct_frpm', 'AP_Count',
                                                 "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

# ✅ ONLY THIS PART CHANGED: Jenks classification replaces normalization
import mapclassify
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col] = classifier.find_bin(gdf[col])


import matplotlib as mpl

# ca map
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

fig, axes = plt.subplots(2, 2, figsize=(10, 10))


axes = axes.flatten()

# norm = mpl.colors.Normalize(vmin=0, vmax=100)
cmap = mpl.cm.cividis  # Change this to your preferred colormap
# Use built-in Plasma with alpha blending
cmap = plt.cm.plasma
cmap.set_under(alpha=0.3)  # Fades low-density areas

from matplotlib.colors import LinearSegmentedColormap
colors = ["#d7191c", "#fdae61", "#ffffbf", "#abd9e9", "#2c7bb6"]  # Red → Orange → White → Light Blue → Blue
# Adjusted blues (darker and more saturated)
# Modified Viridis hex codes with clearer separation
viridis_5distinct = [
    "#440154",  # Deep purple (darkest)
    "#3b528b",  # Royal blue
    "#21918c",  # Teal (midpoint)
    "#5ec962",  # Bright green
    "#fde725"   # Lemon yellow (brightest)
]

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "viridis_5step", viridis_5distinct, N=5
)

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
    'AP_Count'],
    ['College Going Rate','Chronic Absenteeim',  'FRPM'   , 'AP Count']
):
    #ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)
    #ca_counties.plot(ax=ax, color='#989898', edgecolor='black', linewidth=0.5,alpha = 0.5) # midium
    ca_counties.plot(ax=ax, color='gainsboro', edgecolor='black', linewidth=0.5,alpha = 0.5) #light gray
    #ca_counties.plot(ax=ax, color='#F5F5F5', edgecolor='black', linewidth=0.5,alpha = 0.5) #light gray


    gdf.plot(column=col, ax=ax, markersize=20, marker='h',
            cmap=cmap, legend=False, alpha = 0.75,
            edgecolor='gray', linewidth=0.2)


    ax.set_axis_off()
    ax.set_title(title,loc='left')

# cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
# norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm.set_array([])  # Empty array for ScalarMappable


norm = mpl.colors.BoundaryNorm(np.linspace(0, 5, n_classes+1), cmap.N)
cax = fig.add_axes([0.85, 0.3, 0.03, 0.4])  # colorbar position


cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=norm,
                              boundaries=np.arange(5+1),
                              ticks=np.arange(5)+0.5,
                              spacing='proportional')

#cb.outline.set_linewidth(0)        # drop the thick border
cb.ax.tick_params(which='both', length=0)  # remove all tick‐marks

cb.set_ticklabels(['class 1', 'class 2','class 3','class 4','class 5'])
# cb.set_label('Jenks Natural Breaks\nClassification Ranges', loc='top', labelpad=20)
cax.set_title('Jenks Natural Breaks', y=1.02, pad=10, fontsize=10)

# fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

# plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

plt.tight_layout
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.tiff',dpi=600)

plt.show()


In [ ]:
'''
new code
'''
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

# Ensure numeric data
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns and save to new columns for color mapping
for col in cols:
    gdf[f'{col}_norm'] = (gdf[col] - gdf[col].min(skipna=True)) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Optional: Jenks classification (for comparison or other usage)
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = classifier.find_bin(gdf[col])

# Plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

cmap = mpl.cm.plasma
boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, cmap.N)

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    #ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)
    ca_counties.plot(ax=ax, color='#989898', edgecolor='black', linewidth=0.5,alpha = 0.5)

    # Plot with hexagon markers using normalized values
    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=cmap,
        markersize=25,
        marker='h',  # Hexagon marker
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Add colorbar
cax = fig.add_axes([0.85, 0.3, 0.03, 0.4])
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=cmap, norm=norm,
    boundaries=boundaries,
    ticks=boundaries[:-1] + 0.1,
    spacing='proportional'
)
cb.set_ticklabels(['0.0–0.2', '0.2–0.4', '0.4–0.6', '0.6–0.8', '0.8–1.0'])
cax.set_title('Normalized\nValue Ranges', y=1.02, pad=10, fontsize=10)

# Title and layout
plt.suptitle('Spatial Distribution of Key Features', x=0.45, y=0.93, fontweight='bold')
plt.subplots_adjust(wspace=-0.3, hspace=-0.05)
plt.tight_layout()

# Save output
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()



In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify
from matplotlib.colors import ListedColormap

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

# Ensure numeric data
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns and save to new columns for color mapping
for col in cols:
    gdf[f'{col}_norm'] = (gdf[col] - gdf[col].min(skipna=True)) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Optional: Jenks classification (for comparison or other usage)
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = classifier.find_bin(gdf[col])

# Gradient color map (blue to red via light tones)
gradient_colors = [
    '#035AA6',  # Deep Blue
    '#2E83BF',  # Medium Blue
    '#82B6D3',  # Light Blue
    '#E47D53',  # Coral
    '#DD4124'   # Red-Orange
]
gradient_cmap = ListedColormap(gradient_colors)

# Plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)

    # Plot with hexagon markers using normalized values
    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=gradient_cmap,
        markersize=35,
        marker='h',  # Hexagon marker
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Add colorbar
cax = fig.add_axes([0.86, 0.3, 0.03, 0.4])
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=gradient_cmap, norm=norm,
    boundaries=boundaries,
    ticks=boundaries[:-1] + 0.1,
    spacing='proportional'
)

cb.set_ticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cax.set_title('Normalized\nValue', y=1.02, pad=10, fontsize=10)

# Title and layout
plt.suptitle('Spatial Distribution of Key Features', x=0.45, y=0.93, fontweight='bold')
plt.subplots_adjust(wspace=-0.3, hspace=-0.05)
plt.tight_layout()

# Save output
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()


In [ ]:
'''
final
'''
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify
from matplotlib.colors import ListedColormap

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

# Ensure numeric data
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns and save to new columns for color mapping
for col in cols:
    gdf[f'{col}_norm'] = (gdf[col] - gdf[col].min(skipna=True)) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Optional: Jenks classification (for comparison or other usage)
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = classifier.find_bin(gdf[col])

# Gradient color map (blue → red)
gradient_colors = [
    '#035AA6',  # Deep Blue
    '#2E83BF',  # Medium Blue
    '#82B6D3',  # Light Blue
    '#E47D53',  # Coral
    '#DD4124'   # Red-Orange
]
gradient_cmap = ListedColormap(gradient_colors)

# Set up plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10), constrained_layout=False)
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

# Plot each variable
for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    # ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)
    # ca_counties.plot(ax=ax, color='gainsboro', edgecolor='black', linewidth=0.5,alpha = 0.5)
    ca_counties.plot(ax=ax, color='#989898', edgecolor='black', linewidth=0.5,alpha = 0.5)


    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=gradient_cmap,
        markersize=35,
        marker='h',  # Hexagon
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Add colorbar outside of map area
#fig.subplots_adjust(right=0.83, wspace=0.2, hspace=0.2)  # Leave space for colorbar
cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # [left, bottom, width, height]
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=gradient_cmap, norm=norm,
    boundaries=boundaries,
    ticks=np.round(np.linspace(0, 1, 6), 2),
    spacing='proportional'
)
cb.set_ticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cax.set_title('Normalized\nValue', y=1.02, pad=10, fontsize=10)

plt.tight_layout()

# Title and save
# plt.suptitle('Spatial Distribution of Key Features', x=0.45, y=0.93, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify
from matplotlib.colors import ListedColormap

# ─── ADJUSTABLE PARAMETERS ─────────────────────────────────────────────────────
fig_size     = (12, 10)   # keep overall figure size the same
w_space      = 0.05       # horizontal space between subplots (try 0.00 → 0.2)
h_space      = 0.20       # vertical space between subplots
right_margin = 0.83       # how far left the right edge of subplots goes (for colorbar)
cb_position  = [0.857, 0.30, 0.02, 0.40]  # [left, bottom, width, height] for colorbar
# ────────────────────────────────────────────────────────────────────────────────

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, …)

# Ensure numeric data
cols = ['college_going_rate','ChronicAbsenteeismRate','pct_frpm','AP_Count','pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns
for col in cols:
    gdf[f'{col}_norm'] = (
        gdf[col] - gdf[col].min(skipna=True)
    ) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Jenks classification (optional)
n_classes = 5
for col in cols:
    cls = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = cls.find_bin(gdf[col])

# Gradient colormap
gradient_colors = [
    '#035AA6',  # Deep Blue
    '#2E83BF',  # Medium Blue
    '#82B6D3',  # Light Blue
    '#E47D53',  # Coral
    '#DD4124'   # Red-Orange
]
gradient_cmap = ListedColormap(gradient_colors)

# ─── Plot setup ────────────────────────────────────────────────────────────────
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=fig_size, constrained_layout=False)
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

for ax, (col, title) in zip(
    axes,
    zip(
        ['college_going_rate','ChronicAbsenteeismRate','pct_frpm','AP_Count'],
        ['College Going Rate','Chronic Absenteeism','FRPM','AP Count']
    )
):
    ca_counties.plot(
        ax=ax,
        color='#989898',        # a bit darker gray than Gainsboro
        edgecolor='black',
        linewidth=0.5,
        alpha=0.5
    )
    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=gradient_cmap,
        markersize=35,
        marker='h',
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )
    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Adjust spacing (use the parameters above)
fig.subplots_adjust(left=0.05,
                    right=right_margin,
                    bottom=0.08,
                    top=0.92,
                    wspace=w_space,
                    hspace=h_space)

# Colorbar
cax = fig.add_axes(cb_position)
cb = mpl.colorbar.ColorbarBase(
    cax,
    cmap=gradient_cmap,
    norm=norm,
    boundaries=boundaries,
    ticks=np.round(np.linspace(0, 1, 6), 2),
    spacing='proportional'
)
cb.set_ticklabels(['0.0','0.2','0.4','0.6','0.8','1.0'])
cax.set_title('Normalized\nValue', y=1.02, pad=10, fontsize=10)

# Title and save
plt.suptitle('Spatial Distribution of Key Features',
             x=0.45, y=0.95, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()


In [ ]:
'''
make the panel closer
'''
# Set up plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(12.5, 10), constrained_layout=False)  # Slightly wider
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

cmap = mpl.cm.plasma


# Plot each variable
for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    #ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)
    ca_counties.plot(ax=ax, color='gainsboro', edgecolor='black', linewidth=0.5,alpha = 0.5)


    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=cmap,  # gradient_cmap
        markersize=35,
        marker='h',
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Adjust layout to bring plots closer and keep space for colorbar
fig.subplots_adjust(left=0.05, right=0.82, bottom=0.08, top=0.90, wspace=0.05, hspace=0.05)

# Add colorbar
cax = fig.add_axes([0.78, 0.3, 0.02, 0.5])  # [left, bottom, width, height]
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=gradient_cmap, norm=norm,
    boundaries=boundaries,
    ticks=np.round(np.linspace(0, 1, 6), 2),
    spacing='proportional'
)
cb.set_ticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cax.set_title('Normalized\nValue', y=1.02, pad=10, fontsize=10)

# Title and save
plt.suptitle('Spatial Distribution of Key Features', x=0.43, y=0.95, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()


In [ ]:
'''
all years

with Jenks Natural Breaks classification

'''
colormaps = [mpl.cm.viridis, mpl.cm.plasma, mpl.cm.inferno, mpl.cm.cividis, mpl.cm.YlOrRd]

for cmap in colormaps:

#   gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')


  cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
  gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')


  # shap_sp.plot(column = 'shap_ChronicAbsenteeismRate',cmap = "PiYG")
  # zi = (xi – min(x)) / (max(x) – min(x))
  gdf[['college_going_rate',
              'ChronicAbsenteeismRate',
              'pct_frpm', 'AP_Count',
              "pct_passed_math"]] = gdf[['college_going_rate',
                                                      'ChronicAbsenteeismRate',
                                                      'pct_frpm', 'AP_Count',
                                                  "pct_passed_math"]].apply(lambda x: (x - x.min(skipna=True)) / (x.max(skipna=True) - x.min(skipna=True)))

  # ✅ ONLY THIS PART CHANGED: Jenks classification replaces normalization
  import mapclassify
  n_classes = 5
  for col in cols:
      classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
      gdf[col] = classifier.find_bin(gdf[col])


  import matplotlib as mpl

  # ca map
  ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')

  plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})

  fig, axes = plt.subplots(2, 2, figsize=(10, 10))


  axes = axes.flatten()

  for ax, col, title in zip(
      axes[:4],
      ['college_going_rate','ChronicAbsenteeismRate','pct_frpm',
      'AP_Count'],
      ['College Going Rate','Chronic Absenteeim',  'FRPM'   , 'AP Count']
  ):
      ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white',linewidth = 0.35)

      gdf.plot(column=col, ax=ax, markersize=10,
              cmap=cmap, legend=False, alpha = 0.75,
              edgecolor='k', linewidth=0.2)


      ax.set_axis_off()
      ax.set_title(title,loc='left')

  cax = fig.add_axes([0.85, 0.3, 0.02, 0.4])  # Position of the colorbar: [left, bottom, width, height]
  norm = mpl.colors.Normalize(vmin=0, vmax=1)  # Normalized range from 0 to 1
  sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
  sm.set_array([])  # Empty array for ScalarMappable
  fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Value')

  plt.suptitle('Spatial Distribution of Key Features',x=0.45,y=0.93,fontweight='bold')

  plt.subplots_adjust( wspace=-0.3, hspace=-0.05)

  plt.tight_layout
  plt.show()


In [ ]:
'''
new code
'''
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

# Ensure numeric data
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns and save to new columns for color mapping
for col in cols:
    gdf[f'{col}_norm'] = (gdf[col] - gdf[col].min(skipna=True)) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Optional: Jenks classification (for comparison or other usage)
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = classifier.find_bin(gdf[col])

# Gradient color map (blue → red)
gradient_colors = [
    '#035AA6',  # Deep Blue
    '#2E83BF',  # Medium Blue
    '#82B6D3',  # Light Blue
    '#E47D53',  # Coral
    '#DD4124'   # Red-Orange
]
gradient_cmap = ListedColormap(gradient_colors)

# Plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    # ca_counties.plot(ax=ax, color='darkgrey', edgecolor='white', linewidth=0.35)
    ca_counties.plot(ax=ax, color='#989898', edgecolor='black', linewidth=0.5,alpha = 0.5)

    # Plot with hexagon markers using normalized values
    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=gradient_cmap,
        markersize=35,
        marker='h',  # Hexagon marker
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Add colorbar
cax = fig.add_axes([0.82, 0.3, 0.03, 0.4])
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=gradient_cmap, norm=norm,
    boundaries=boundaries,
    ticks=np.round(np.linspace(0, 1, 6), 2),
    spacing='proportional'
)
cb.set_ticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cax.set_title('Normalized\nValue Ranges', y=1.02, pad=10, fontsize=10)

# Title and layout
# plt.suptitle('Spatial Distribution of Key Features', x=0.45, y=0.93, fontweight='bold')
plt.subplots_adjust(wspace=-0.3, hspace=-0.05)
plt.tight_layout()

# Save output
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()



In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import mapclassify
from matplotlib.colors import ListedColormap

# Load data
ca_counties = gpd.read_file(f'{DATA_DIR}/geographic_boundaries/counties').to_crs('EPSG:4326')
# gdf = gpd.GeoDataFrame(school_related, geometry=gpd.points_from_xy(school_related['Longitude'], school_related['Latitude']), crs='EPSG:4326')

# Ensure numeric data
cols = ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count', 'pct_passed_math']
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# Normalize columns and save to new columns for color mapping
for col in cols:
    gdf[f'{col}_norm'] = (
        gdf[col] - gdf[col].min(skipna=True)
    ) / (gdf[col].max(skipna=True) - gdf[col].min(skipna=True))

# Optional: Jenks classification (for comparison or other usage)
n_classes = 5
for col in cols:
    classifier = mapclassify.NaturalBreaks(gdf[col].dropna(), k=n_classes)
    gdf[col + '_jenks'] = classifier.find_bin(gdf[col])

# Gradient color map (blue → red)
gradient_colors = [
    '#035AA6',  # Deep Blue
    '#2E83BF',  # Medium Blue
    '#82B6D3',  # Light Blue
    '#E47D53',  # Coral
    '#DD4124'   # Red-Orange
]
gradient_cmap = ListedColormap(gradient_colors)

# Plotting
plt.rc('axes.spines', **{'bottom': False, 'left': False, 'right': False, 'top': False})
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

boundaries = np.linspace(0, 1, n_classes + 1)
norm = mpl.colors.BoundaryNorm(boundaries, gradient_cmap.N)

for ax, col, title in zip(
    axes[:4],
    ['college_going_rate', 'ChronicAbsenteeismRate', 'pct_frpm', 'AP_Count'],
    ['College Going Rate', 'Chronic Absenteeism', 'FRPM', 'AP Count']
):
    ca_counties.plot(ax=ax, color='#989898', edgecolor='black', linewidth=0.5, alpha=0.5)

    # Plot with hexagon markers using normalized values
    gdf.dropna(subset=[f'{col}_norm']).plot(
        ax=ax,
        column=f'{col}_norm',
        cmap=gradient_cmap,
        markersize=35,
        marker='h',  # Hexagon marker
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8,
        legend=False
    )

    ax.set_axis_off()
    ax.set_title(title, loc='left')

# Add colorbar (moved right so it no longer overlaps panels)
cax = fig.add_axes([0.92, 0.3, 0.03, 0.4])  # <-- x-position changed from 0.82 to 0.92
cb = mpl.colorbar.ColorbarBase(
    cax, cmap=gradient_cmap, norm=norm,
    boundaries=boundaries,
    ticks=np.round(np.linspace(0, 1, 6), 2),
    spacing='proportional'
)
cb.set_ticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cax.set_title('Normalized\nValue Ranges', y=1.02, pad=10, fontsize=10)

# Title and layout
# plt.suptitle('Spatial Distribution of Key Features', x=0.45, y=0.93, fontweight='bold')
plt.subplots_adjust(wspace=-0.3, hspace=-0.05)
plt.tight_layout()

# Save output
plt.savefig(f'{OUTPUT_DIR}/plots/keyfeatures.png', dpi=600)
plt.show()
